# 02 — Shuffle partitions: cómo elegir el número correcto

`spark.sql.shuffle.partitions=200` es el default. Es una locura para
muchos casos. Acá aprendés a dimensionarlo bien.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("04_02_shuffle_partitions")
    .master("spark://spark-master:7077")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

df = spark.read.parquet("/home/jovyan/work/data/sample/ventas/parquet_limpio")
print("Filas:", df.count())

## La regla: target ~100-200 MB por partición post-shuffle

In [ ]:
import time

def medir(partitiones, df):
    spark.conf.set("spark.sql.shuffle.partitions", str(partitiones))
    t = time.time()
    df.groupBy("ciudad").agg(F.avg("precio"), F.count("*")).collect()
    elapsed = time.time() - t
    return elapsed

for n in [1, 4, 8, 32, 200, 1000]:
    t = medir(n, df)
    print(f"{n:>4} particiones → {t:.2f}s")

## Cómo se ven los stages con pocas vs muchas particiones

En la Spark UI:

- **200 particiones, dataset chico**: vas a ver 200 tasks, casi todos
  leyendo < 1 MB. Overhead de scheduling domina.
- **8 particiones, dataset chico**: 8 tasks, cada uno con trabajo
  serio. Aprovecha el paralelismo real sin over-subscripción.
- **8 particiones, dataset de 500 GB**: 8 tasks con 60 GB cada uno.
  Out of memory, GC pauses, desastre.

**Heurística rápida**:
```
n_particiones = max(2, ceil(datos_post_shuffle_mb / 200))
```

In [ ]:
import math

def particiones_recomendadas(datos_mb, target_mb_por_particion=200):
    return max(2, math.ceil(datos_mb / target_mb_por_particion))

ejemplos = [
    ('Dataset chico', 50),
    ('Dataset mediano', 2000),
    ('Dataset grande', 100000),
    ('Dataset enorme', 2000000),
]
for nombre, mb in ejemplos:
    print(f'{nombre} ({mb} MB) -> {particiones_recomendadas(mb)} particiones')

## AQE al rescate

Si no querés pensar, AQE coalesce en runtime. Con AQE prendido, el
default de 200 no es tan terrible — Spark va a unir las particiones
vacías.

Sin AQE, mal dimensionado = plata quemada en el cluster.

## ✅ Checkpoint

- [x] Sé la regla de 100-200 MB por partición.
- [x] Sé calcular particiones recomendadas según tamaño.
- [x] Sé que AQE atenúa el problema si lo dejas prendido.

**Siguiente**: `03_skew_salting.ipynb` — el problema más común y más
doloroso.